In [0]:
%sql
-- create catalog tasks ;
-- create schema tasks_db;
-- use catalog tasks ;
use schema tasks_db;

In [0]:
%sql
create volume v_files ;


In [0]:
dbutils.fs.mkdirs('/Volumes/dkishore/tasks_db/v_files/csv')

True

In [0]:
display(dbutils.fs.ls('/Volumes/dkishore/tasks_db/v_files/csv'))

path,name,size,modificationTime
dbfs:/Volumes/dkishore/tasks_db/v_files/csv/ord.csv,ord.csv,554,1787146608000


In [0]:
%sql
select current_schema()

current_schema()
tasks_db


1. rdd won't support volumes 
2. Thats why i found  3 ways to use them

In [0]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('SparkByExamples.com').getOrCreate()
sc=spark.sparkContext

In [0]:
rdd=spark.read.text('/Volumes/dkishore/tasks_db/v_files/csv/original_orders.csv').rdd
# rdd.collect()
raw_data=rdd.map(lambda x:x[0]).collect()
ord_rdd=sc.parallelize(raw_data)
# ord_rdd.collect()
h=ord_rdd.first()
f_rdd=ord_rdd.filter(lambda x:x!=h).map(lambda x:x.split(","))
f_rdd.collect()

[Row(value='order_id,customer_id,product_id,category,quantity,unit_price,order_date,city,order_status'),
 Row(value='O1001,C101,P501,Electronics,2,25000,2026-08-01,Ahmedabad,COMPLETED'),
 Row(value='O1002,C102,P502,Fashion,3,1500,2026-08-01,Mumbai,COMPLETED'),
 Row(value='O1003,C101,P503,Electronics,1,50000,2026-08-02,Ahmedabad,PENDING'),
 Row(value='O1004,C103,P504,Grocery,5,500,2026-08-02,Pune,COMPLETED'),
 Row(value='O1005,C104,P505,Fashion,2,2000,2026-08-03,Delhi,CANCELLED')]

In [0]:
f_rdd.take(5)
f_rdd.count()
f_rdd.getNumPartitions()

[['O1001',
  'C101',
  'P501',
  'Electronics',
  '2',
  '25000',
  '2026-08-01',
  'Ahmedabad',
  'COMPLETED'],
 ['O1002',
  'C102',
  'P502',
  'Fashion',
  '3',
  '1500',
  '2026-08-01',
  'Mumbai',
  'COMPLETED'],
 ['O1003',
  'C101',
  'P503',
  'Electronics',
  '1',
  '50000',
  '2026-08-02',
  'Ahmedabad',
  'PENDING'],
 ['O1004',
  'C103',
  'P504',
  'Grocery',
  '5',
  '500',
  '2026-08-02',
  'Pune',
  'COMPLETED'],
 ['O1005',
  'C104',
  'P505',
  'Fashion',
  '2',
  '2000',
  '2026-08-03',
  'Delhi',
  'CANCELLED']]

### RDDD

rdd is a Low‑level distributed collection of objects. Each record is just a list or string until you manually parse it.It is used for basic and testing purpose transformations.

It uses mapreduce , groupby ,map ,etc .

Pros: Fine‑grained control, useful for custom transformations.

Cons: Verbose, no automatic schema, no query optimizations.

syntax :- 

rdd = spark.read.text("/Volumes/.../ord.csv").rdd.map(lambda row: row[0]) <br>
header = rdd.first() <br>
parsed_rdd = rdd.filter(lambda line: line != header).map(lambda line: line.split(","))


### DF

spark dataframe is a  Higher‑level abstraction built on RDDs, with rows & columns like a table. Backed by Spark’s Catalyst optimizer.It is used for the intermediate and high level transformations.

It uses groupby ,select ,agg like sql 

Pros: Easier syntax, optimized execution, integrates with SQL, MLlib, and structured APIs.

Cons: Less control than raw RDDs for unusual transformations.

syntax :- spark.read.csv(....,header=True,inferSchema=True)

In [0]:
f_rdd.take(3)

['O1001,C101,P501,Electronics,2,25000,2026-08-01,Ahmedabad,COMPLETED',
 'O1002,C102,P502,Fashion,3,1500,2026-08-01,Mumbai,COMPLETED',
 'O1003,C101,P503,Electronics,1,50000,2026-08-02,Ahmedabad,PENDING']

In [0]:
# 1. Filter orders having order_status = "COMPLETED".
# 2. Extract category, quantity, and unit_price.
# 3. Calculate order_amount as:
# order_amount = quantity  unit_price
res1=f_rdd.filter(lambda x:x[8]=="COMPLETED").map(lambda x:(x[3],int(x[4])*int(x[5])))
res1.collect()

[('Electronics', 50000), ('Fashion', 4500), ('Grocery', 2500)]

In [0]:
# Task 3 — Aggregation Using RDD
res2=f_rdd.map(lambda x:(x[3],int(x[4])*int(x[5])))
res2.collect()

[('Electronics', 50000),
 ('Fashion', 4500),
 ('Electronics', 50000),
 ('Grocery', 2500),
 ('Fashion', 4000)]

In [0]:
res2.reduceByKey(lambda x,y:x+y).collect()

[('Electronics', 100000), ('Grocery', 2500), ('Fashion', 8500)]

In [0]:
%sql
select * from orders_view

order_id,customer_id,product_id,category,quantity,unit_price,order_date,city,order_status
O1001,C101,P501,Electronics,2,25000,2026-08-01,Ahmedabad,COMPLETED
O1002,C102,P502,Fashion,3,1500,2026-08-01,Mumbai,COMPLETED
O1003,C101,P503,Electronics,1,50000,2026-08-02,Ahmedabad,PENDING
O1004,C103,P504,Grocery,5,500,2026-08-02,Pune,COMPLETED
O1005,C104,P505,Fashion,2,2000,2026-08-03,Delhi,CANCELLED


In [0]:

df=spark.read.csv('/Volumes/dkishore/tasks_db/v_files/csv/original_orders.csv',header=True,inferSchema=True)
df.createOrReplaceTempView('orders_view')
# df.write.format('delta').createOrReplaceTempView('orders_view')

In [0]:
display(spark.sql('select category,sum(quantity*unit_price) as total from orders_view group by category'))
display(spark.sql('select customer_id,sum(quantity*unit_price) as total from orders_view where order_status ="COMPLETED" group by customer_id order by total desc'))
display(spark.sql('select city, sum(quantity*unit_price) as total from orders_view group by 1 order by 2 desc limit 1'))
display(spark.sql('select order_status,count(*) as cnt from orders_view group by order_status'))
display(spark.sql('select customer_id,sum(quantity*unit_price) as total  from orders_view group by customer_id having total>50000'))

category,total
Fashion,8500
Grocery,2500
Electronics,100000


customer_id,total
C101,50000
C102,4500
C103,2500


city,total
Ahmedabad,100000


order_status,cnt
CANCELLED,1
COMPLETED,3
PENDING,1


customer_id,total
C101,100000


In [0]:
# Part C — Delta Table & Delta Operations
 
# Task 6 — Create a Delta Table
 
 
# Store the cleaned order DataFrame as a Delta table named:
 
# retail_orders
df.write.format('delta').saveAsTable('retailed_orders')
display(spark.sql('desc detail retailed_orders'))

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,3dd75864-e9c6-4178-97b6-7e91637cceac,dkishore.tasks_db.retailed_orders,null,abfss://unity-catalog-storage@dbstoragerlufpmb73urdq.dfs.core.windows.net/7405616050946422/__unitystorage/catalogs/fc26a123-8d15-4d3a-a4bf-05c4fbb54c03/tables/e6b9cfb8-5112-4c6f-aac2-257a86fc1627,2026-08-19T17:04:06.896Z,2026-08-19T17:04:11Z,List(),List(),1,2921,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql

insert into retailed_orders values('2001','C201','P601','Electronics',1,75000,'2026-08-10','Surat','COMPLETED')

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
-- # Task 8 — UPDATE
update retailed_orders set order_status = 'COMPLETED' where order_id = 'O1003' ;

num_affected_rows
1


In [0]:
%sql
-- Task 9 — DELETE
delete from retailed_orders where order_id = 'O1005';

num_affected_rows
1


In [0]:
%sql
desc history retailed_orders;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-08-19T17:09:08Z,144097337326174,fy26databricks5thaugto7thsepuser157@acp11792b.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2024741413124273),0819-052919-t19tzpc0,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2987, p25FileSize -> 2915, numDeletionVectorsRemoved -> 1, minFileSize -> 2915, numAddedFiles -> 1, maxFileSize -> 2915, p75FileSize -> 2915, p50FileSize -> 2915, numAddedBytes -> 2915)",null,Databricks-Runtime/17.3.x-cpu-ml-photon-scala2.13
4,2026-08-19T17:09:06Z,144097337326174,fy26databricks5thaugto7thsepuser157@acp11792b.onmicrosoft.com,DELETE,"Map(predicate -> [""(order_id#3656 = O1005)""])",null,List(2024741413124273),0819-052919-t19tzpc0,3,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1687, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1285, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 401)",null,Databricks-Runtime/17.3.x-cpu-ml-photon-scala2.13
3,2026-08-19T17:08:33Z,144097337326174,fy26databricks5thaugto7thsepuser157@acp11792b.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2024741413124273),0819-052919-t19tzpc0,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 8037, p25FileSize -> 2987, numDeletionVectorsRemoved -> 1, minFileSize -> 2987, numAddedFiles -> 1, maxFileSize -> 2987, p75FileSize -> 2987, p50FileSize -> 2987, numAddedBytes -> 2987)",null,Databricks-Runtime/17.3.x-cpu-ml-photon-scala2.13
2,2026-08-19T17:08:29Z,144097337326174,fy26databricks5thaugto7thsepuser157@acp11792b.onmicrosoft.com,UPDATE,"Map(predicate -> [""(order_id#2995 = O1003)""])",null,List(2024741413124273),0819-052919-t19tzpc0,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 6596, numDeletionVectorsUpdated -> 0, scanTimeMs -> 4108, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 2598, rewriteTimeMs -> 2453)",null,Databricks-Runtime/17.3.x-cpu-ml-photon-scala2.13
1,2026-08-19T17:06:56Z,144097337326174,fy26databricks5thaugto7thsepuser157@acp11792b.onmicrosoft.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2024741413124273),0819-052919-t19tzpc0,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 2518)",null,Databricks-Runtime/17.3.x-cpu-ml-photon-scala2.13
0,2026-08-19T17:04:11Z,144097337326174,fy26databricks5thaugto7thsepuser157@acp11792b.onmicrosoft.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2024741413124273),0819-052919-t19tzpc0,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 5, numOutputBytes -> 2921)",null,Databricks-Runtime/17.3.x-cpu-ml-photon-scala2.13


In [0]:
%sql
create table order_updates as select * from retailed_orders;


num_affected_rows,num_inserted_rows


In [0]:
%sql
alter table retailed_orders
add columns order_amount int ;

In [0]:
%sql
-- Task 10 — MERGE / UPSERT
 
merge into retailed_orders t
using order_updates s
on t.order_id = s.order_id
when matched then update set 
 t.order_id=s.order_id,
 t.customer_id=s.customer_id,
 t.product_id=s.product_id,
 t.category=s.category,
 t.quantity=s.quantity,
 t.unit_price=s.unit_price,
 t.order_date=s.order_date,
 t.city=s.city,
 t.order_status=s.order_status
,t.order_amount=s.quantity*s.unit_price
when not matched then insert (order_id,customer_id,product_id,category,quantity,unit_price,order_date,city,order_status,order_amount) values (s.order_id,s.customer_id,s.product_id,s.category,s.quantity,s.unit_price,s.order_date,s.city,s.order_status,s.quantity*s.unit_price)

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,4,0,1


In [0]:
%sql
select * from retailed_orders where order_id = 'O1002';

order_id,customer_id,product_id,category,quantity,unit_price,order_date,city,order_status,order_amount
O1002,C102,P502,Fashion,3,1500,2026-08-01,Mumbai,COMPLETED,4500


In [0]:
%sql
insert into order_updates values('O1006','C102','P502','Electronics',4,2000,'2026-08-10','Kakinada','COMPLETED');
update order_updates
set unit_price=2000 where order_id='O1002';

num_affected_rows
1


In [0]:
%sql
select * from order_updates

order_id,customer_id,product_id,category,quantity,unit_price,order_date,city,order_status
O1002,C102,P502,Fashion,3,2000,2026-08-01,Mumbai,COMPLETED
O1003,C101,P503,Electronics,1,50000,2026-08-02,Ahmedabad,COMPLETED
O1001,C101,P501,Electronics,2,25000,2026-08-01,Ahmedabad,COMPLETED
O1004,C103,P504,Grocery,5,500,2026-08-02,Pune,COMPLETED
O1006,C102,P502,Electronics,4,2000,2026-08-10,Kakinada,COMPLETED


After this re-run merge cell again . Observe the records in cell 30 and cell 26 , U see the change of order_amount for order_id = O1002 and also the newly inserted value too 

In [0]:
%sql
select * from retailed_orders ;

order_id,customer_id,product_id,category,quantity,unit_price,order_date,city,order_status,order_amount
O1002,C102,P502,Fashion,3,2000,2026-08-01,Mumbai,COMPLETED,6000
O1003,C101,P503,Electronics,1,50000,2026-08-02,Ahmedabad,COMPLETED,50000
O1001,C101,P501,Electronics,2,25000,2026-08-01,Ahmedabad,COMPLETED,50000
O1004,C103,P504,Grocery,5,500,2026-08-02,Pune,COMPLETED,2500
O1006,C102,P502,Electronics,4,2000,2026-08-10,Kakinada,COMPLETED,8000


In [0]:
%sql
-- Task 11 — Delta Time Travel
desc history retailed_orders;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
10,2026-08-19T17:52:39Z,144097337326174,fy26databricks5thaugto7thsepuser157@acp11792b.onmicrosoft.com,MERGE,"Map(predicate -> [""(order_id#8597 = order_id#8616)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2024741413124273),0819-052919-t19tzpc0,9,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 3144, numTargetBytesRemoved -> 3107, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 4, executionTimeMs -> 1955, materializeSourceTimeMs -> 1, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 916, numTargetRowsUpdated -> 4, numOutputRows -> 5, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 985)",null,Databricks-Runtime/17.3.x-cpu-ml-photon-scala2.13
9,2026-08-19T17:50:03Z,144097337326174,fy26databricks5thaugto7thsepuser157@acp11792b.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2024741413124273),0819-052919-t19tzpc0,8,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 3107, p25FileSize -> 3107, numDeletionVectorsRemoved -> 1, minFileSize -> 3107, numAddedFiles -> 1, maxFileSize -> 3107, p75FileSize -> 3107, p50FileSize -> 3107, numAddedBytes -> 3107)",null,Databricks-Runtime/17.3.x-cpu-ml-photon-scala2.13
8,2026-08-19T17:50:02Z,144097337326174,fy26databricks5thaugto7thsepuser157@acp11792b.onmicrosoft.com,DELETE,"Map(predicate -> [""(order_id#7025 = 2001)""])",null,List(2024741413124273),0819-052919-t19tzpc0,7,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 765, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 484, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 281)",null,Databricks-Runtime/17.3.x-cpu-ml-photon-scala2.13
7,2026-08-19T17:42:00Z,144097337326174,fy26databricks5thaugto7thsepuser157@acp11792b.onmicrosoft.com,MERGE,"Map(predicate -> [""(order_id#4888 = order_id#4898)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2024741413124273),0819-052919-t19tzpc0,6,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 3107, numTargetBytesRemoved -> 2915, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 4903, materializeSourceTimeMs -> 4, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1636, numTargetRowsUpdated -> 5, numOutputRows -> 5, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 3223)",null,Databricks-Runtime/17.3.x-cpu-ml-photon-scala2.13
6,2026-08-19T17:41:47Z,144097337326174,fy26databricks5thaugto7thsepuser157@acp11792b.onmicrosoft.com,ADD COLUMNS,"Map(columns -> [{""column"":{""name"":""order_amount"",""type"":""integer"",""nullable"":true,""metadata"":{}}}])",null,List(2024741413124273),0819-052919-t19tzpc0,5,WriteSerializable,true,Map(),null,Databricks-Runtime/17.3.x-cpu-ml-photon-scala2.13
5,2026-08-19T17:09

In [0]:
%sql
-- select * from retailed_orders version as of 10 ;
-- restore retailed_orders version as of 8;
select order_id ,order_amount ,'present' as current_values from retailed_orders version as of 10 
union all
select order_id ,order_amount,'previous' as past_values  from retailed_orders version as of 9
order by order_id

order_id,order_amount,current_values
O1001,50000,present
O1002,4500,previous
O1002,6000,present
O1003,50000,present
O1004,2500,present
O1006,8000,present


In [0]:
df_stream=(
    spark.readStream.format('cloudFiles')
    .option('cloudFiles.format','csv')
    .option('header','true')
    .option('cloudFiles.schemaLocation','/Volumes/dkishore/tasks_db/v_files/stream')
    .option('cloudFiles.inferColumnTypes','true')
    .option('mergeSchema','true')
    .load('/Volumes/dkishore/tasks_db/v_files/csv/')
)
df_stream.writeStream.format('delta').option('checkpointLocation','/Volumes/dkishore/tasks_db/v_files/stream').outputMode('append').toTable('retailOrders_streaming')

In [0]:
%sql
-- Write an SQL query on retailOrders_streaming to retrieve the total completed sales amount for each product category.

select category,sum(quantity*unit_price) as total_sales_amount from retailorders_streaming where order_status='COMPLETED' group by category ;

category,total_sales_amount
Fashion,4500
Grocery,2500
Electronics,50000


In [0]:
%sql
select date_format(order_date,'yyyy-MM') as revenue_month,sum(quantity*unit_price) as total_revenue from retailorders_streaming where order_status='COMPLETED' group by 1 order by revenue_month

revenue_month,total_revenue
2026-06,103000
2026-07,136200
2026-08,57000


## REFERENCES 

In [0]:
# RDD actions on UC Volumes files are not supported; collect via DataFrame then parallelize
rows = [row[0] for row in spark.read.text('/Volumes/dkishore/tasks_db/v_files/csv/ord.csv').collect()]
header = rows[0]
ord_rdd = sc.parallelize([row.split(',') for row in rows[1:]])
ord_rdd.take(5)  # Ensure the data source is accessible and mounted correctly

[['1001', 'C101', '2026-08-10', 'Ahmedabad', 'UPI', 'PLACED', '850'],
 ['1002', 'C102', '2026-08-10', 'Mumbai', 'CARD', 'SHIPPED', '1200'],
 ['1003', 'C103', '2026-08-11', 'Delhi', 'COD', 'DELIVERED', '650'],
 ['1004', 'C104', '2026-08-11', 'Ahmedabad', 'UPI', 'CANCELLED', '900'],
 ['1005', 'C105', '2026-08-12', 'Pune', 'CARD', 'DELIVERED', '1450']]

In [0]:
# Using RDD operations only:
 
# 1. Filter orders having order_status = "COMPLETED".
# 2. Extract category, quantity, and unit_price.
# 3. Calculate order_amount as:
 
# order_amount = quantity  unit_price
rdd.filter(lambda x:x[5]=='COMPLETED').map(lambda x:(x[4],int(x[6])))

In [0]:
with open('/Volumes/dkishore/tasks_db/v_files/csv/original_orders.csv', 'r') as f:
    lines = f.readlines()
# print(lines)
# Convert list to distributed RDD
raw_rdd = sc.parallelize(lines)

header = raw_rdd.first()
rdd = raw_rdd.filter(lambda x: x != header).map(lambda x: x.split('\n'))

# rdd.collect()
rdd.collect()

[['O1001,C101,P501,Electronics,2,25000,2026-08-01,Ahmedabad,COMPLETED', ''],
 ['O1002,C102,P502,Fashion,3,1500,2026-08-01,Mumbai,COMPLETED', ''],
 ['O1003,C101,P503,Electronics,1,50000,2026-08-02,Ahmedabad,PENDING', ''],
 ['O1004,C103,P504,Grocery,5,500,2026-08-02,Pune,COMPLETED', ''],
 ['O1005,C104,P505,Fashion,2,2000,2026-08-03,Delhi,CANCELLED']]

In [0]:
# On Personal (Assigned) compute, RDD operations are fully supported
rdd = spark.read.text('/Volumes/dkishore/tasks_db/v_files/csv/original_orders.csv').rdd
header = rdd.first()
parsed_rdd = rdd.filter(lambda line: line != header).map(lambda x:x[0])
parsed_rdd.collect()

['O1001,C101,P501,Electronics,2,25000,2026-08-01,Ahmedabad,COMPLETED',
 'O1002,C102,P502,Fashion,3,1500,2026-08-01,Mumbai,COMPLETED',
 'O1003,C101,P503,Electronics,1,50000,2026-08-02,Ahmedabad,PENDING',
 'O1004,C103,P504,Grocery,5,500,2026-08-02,Pune,COMPLETED',
 'O1005,C104,P505,Fashion,2,2000,2026-08-03,Delhi,CANCELLED']